In [ ]:
!pip install xgboost -q

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

from xgboost import XGBClassifier

os.makedirs("plots", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content/drive/MyDrive/Colab\ Notebooks/fraud-detection


In [ ]:
df = pd.read_csv("creditcard.csv")
df.head()

In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
ratio = (y_train == 0).sum() / (y_train == 1).sum()
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=3000))
    ]),
    "Random Forest": RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
),

"XGBoost": XGBClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42
)
}


results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_probs = model.predict_proba(X_test)[:, 1]

    roc = roc_auc_score(y_test, y_probs)
    pr = average_precision_score(y_test, y_probs)

    results.append([name, roc, pr])

results_df = pd.DataFrame(results, columns=["Model", "ROC_AUC", "PR_AUC"])
results_df

In [ ]:
results_df.to_csv("results/baseline_results.csv", index=False)

In [ ]:
def plot_roc_curve(model_name, y_true, y_probs):
    fpr, tpr, _ = roc_curve(y_true, y_probs)

    plt.figure()
    plt.plot(fpr, tpr, label=f"{model_name}")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {model_name}")
    plt.legend()
    plt.savefig(f"plots/roc_{model_name}.png")
    plt.show()

In [ ]:
def plot_pr_curve(model_name, y_true, y_probs):
    precision, recall, _ = precision_recall_curve(y_true, y_probs)

    plt.figure()
    plt.plot(recall, precision, label=f"{model_name}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision-Recall Curve - {model_name}")
    plt.legend()
    plt.savefig(f"plots/pr_{model_name}.png")
    plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

def plot_confusion_heatmap(model_name, y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure()
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.savefig(f"plots/cm_{model_name}.png")
    plt.show()

In [ ]:
def plot_prob_dist(model_name, y_probs, y_true):
    plt.figure()

    plt.hist(y_probs[y_true == 0], bins=50, alpha=0.6, label="Class 0")
    plt.hist(y_probs[y_true == 1], bins=50, alpha=0.6, label="Class 1")

    plt.xlabel("Predicted Probability")
    plt.ylabel("Count")
    plt.title(f"Probability Distribution - {model_name}")
    plt.legend()
    plt.savefig(f"plots/prob_dist_{model_name}.png")
    plt.show()

In [ ]:
for name, model in models.items():
    model.fit(X_train, y_train)

    y_probs = model.predict_proba(X_test)[:, 1]
    y_preds = model.predict(X_test)

    plot_roc_curve(name, y_test, y_probs)
    plot_pr_curve(name, y_test, y_probs)
    plot_confusion_heatmap(name, y_test, y_preds)
    plot_prob_dist(name, y_probs, y_test)